In [10]:
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url="https://api.groq.com/openai/v1")

In [11]:
def search_kb(query: str):
    response = requests.post(
                    "http://localhost:8000/search", json={"query": query, "limit": 3}
    )
    return response.json()

In [12]:
tools = [
    {
        "type": "function",
        "name": "search_kb",
        "description": "Busca informações na base de conhecimento para responder perguntas.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string", "description": "A pergunta do usuário"
                },
            },
            "required": ["query"],
        },
    },
]

input_list = [{"role": "user", "content": "what are AAPL main financial risks?"}]

In [13]:
response = client.responses.parse(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    tools=tools,
    input=input_list,
)

input_list += response.output

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = search_kb(**args)
        
        texts = [r["text"] for r in result["results"]]

        input_list.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({"results": texts}, ensure_ascii=False),
            }
        )

In [14]:
final_response = client.responses.parse(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    instructions="Responda a pergunta do usuário usando as informações retornadas pela busca",
    tools=tools,
    input=input_list,
)

print(final_response.output_text)

AAPL's main financial risks include macroeconomic and industry risks, financial risks, and general risks. Some specific risks include interest rate risk, credit risk on trade accounts receivable, vendor non-trade receivables, and prepayments related to long-term supply agreements. Additionally, AAPL faces risks related to new business strategies, commercial relationships, and acquisitions, such as distraction of management, greater-than-expected liabilities and expenses, and inadequate return on capital. The company is also subject to specific obligations relating to the collection and processing of data associated with minors and sensitive information, and failure to comply with these rules and requirements can result in litigation, government investigations, and significant fees or fines.
